# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. This workflow will cover loading the Croissant schema, inspecting metadata, discovering and extracting record sets and fields, and performing exploratory data analysis (EDA).

### Dataset Source
The dataset is defined by a Croissant schema at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and access the dataset object using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Initialize the Dataset
dataset = mlc.Dataset(croissant_url)
# Print basic metadata (using attribute access instead of a dict)
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}\nVersion: {dataset.metadata.version}")

## 2. Data Overview

Let's review the dataset's available record sets, fields, and their `@id`s. This will help us discover the structure of the tabular data and what columns/fields are available for analysis.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.metadata.recordSets
print(f"Record sets found: {len(record_sets)}\n")
for rset in record_sets:
    print(f"Record set name: {rset.name}")
    print(f" - @id: {rset['@id']}")
    print(f" - Description: {rset.description}")
    print(f" - Number of fields: {len(rset.fields)}")
    print(" - Fields and their @id's:")
    for field in rset.fields:
        print(f"    - {field.name}: {field['@id']} (Data type: {field.dataType})")
    print()

## 3. Data Extraction

We'll load data from each available record set into pandas DataFrames for further analysis.

Replace `<record_set_id>` with the `@id` of the record set you wish to extract.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"  Failed to load records for {rs_id}: {e}")
    print()
# Example: examine columns from the first (main) record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's apply some simple filtering, normalization, and grouping to a numeric field. Use the field `@id` from section 2.

In [ ]:
# Choose a record set and fields for analysis (replace with actual @id from your dataset overview output above)
# Edit the following example as needed using the detected field names/@ids
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(record_set_id)

# Show which columns are available for numeric analysis
print(f"Columns in {record_set_id}: {df.columns.tolist() if df is not None else 'N/A'}")

# (For demonstration, pick first numeric-looking field below, or edit as relevant)
numeric_fields = [col for col in (df.columns if df is not None else []) if df[col].dtype.kind in 'iufc']
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using '{numeric_field}' for numeric analysis.")

    # Filter for some threshold
    threshold = df[numeric_field].quantile(0.9)  # e.g., top 10%
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f} ({filtered_df.shape[0]} rows):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical field
    group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped average '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationships to a group/categorical field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Explored the Croissant metadata for the Mass Spectrometry EV dataset
- Inspected available record sets and fields via their `@id`
- Loaded tabular data into pandas
- Performed basic numeric filtering, normalization, grouping, and visualization

**Next steps:** Depending on your research question, you may continue with feature engineering, modeling, or more in-depth biological/clinical analysis.

For more information on dataset annotation and curation standards, see the [MLCommons Croissant documentation](https://mlcommons.org/croissant/).
